# breadth vs. depth vs. time

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import nn_bowl
import pandas as pd
import numpy as np
import tqdm

from sklearn.manifold import MDS

import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [3]:
BREADTH_TICK_VALS = [0.0, 1.0, 10.0, 100.0, 1000.0, 10000.0]
BREADTH_TICK_TEXT = ["0", "1", "10", "10<sup>2</sup>", "10<sup>3</sup>", "10<sup>4</sup>"]

DEPTH_TICK_VALS = [0.0, 1.0, 10.0, 100.0, 1000.0, 10000.0]
DEPTH_TICK_TEXT = ["0", "1", "10", "10<sup>2</sup>", "10<sup>3</sup>", "10<sup>4</sup>"]

TIME_TICK_VALS = [0.0, 0.1, 0.2, 0.3, 0.5, 1.0]
TIME_TICK_TEXT = ["0", "0.1", "0.2", "0.3", "0.5", "1"]

In [4]:
cfg_base = nn_bowl.utils.load_config("config_breadth_depth_time_v7.yaml")
WIDTH = cfg_base.plotting.width.full

nn_bowl.utils.set_plotly_template(cfg_base)
figures_dir = nn_bowl.paths.FIGURES_DIR / "breadth_depth_time"

# ensuring that all evaluation files exist for all combinations of lam_depth, lam_breadth, lam_time and instance
nn_bowl.utils.ensure_evaluation_files_exist(cfg_base, skip_pruning=False)

ensuring that all evaluation files exist...


100%|██████████| 648/648 [00:35<00:00, 18.15it/s]


In [5]:
# model df
model_df = nn_bowl.utils.get_model_df(cfg_base)
model_df["lam_breadth"] = model_df["costs.space.lam_breadth"]
model_df["lam_depth"] = model_df["costs.space.lam_depth"]
model_df["lam_time"] = model_df["costs.time.lam"]
model_df

  0%|          | 0/648 [00:00<?, ?it/s]

  0%|          | 2/648 [00:02<09:29,  1.14it/s]

Labels shape for cifar10: (10000,)


100%|██████████| 648/648 [01:11<00:00,  9.08it/s]


,model_name,costs.space.lam_breadth,costs.space.lam_depth,costs.time.lam,instance,cost_depth,cost_breadth,cost_time,accuracy,lam_breadth,lam_depth,lam_time
0,lam_breadth=0.0_lam_depth=0.0_lam=0.0_i0,0.0,0.0,0.0,0,0.130335,1.372590e-01,1.221888,0.8759,0.0,0.0,0.0
1,lam_breadth=0.0_lam_depth=0.0_lam=0.1_i0,0.0,0.0,0.1,0,0.132233,1.377218e-01,0.993981,0.8829,0.0,0.0,0.1
2,lam_breadth=0.0_lam_depth=0.0_lam=0.2_i0,0.0,0.0,0.2,0,0.141518,1.425608e-01,0.677266,0.8739,0.0,0.0,0.2
3,lam_breadth=0.0_lam_depth=0.0_lam=0.3_i0,0.0,0.0,0.3,0,0.138716,1.410775e-01,0.621916,0.8713,0.0,0.0,0.3
4,lam_breadth=0.0_lam_depth=0.0_lam=0.5_i0,0.0,0.0,0.5,0,0.141579,1.363750e-01,0.467292,0.8642,0.0,0.0,0.5
...,...,...,...,...,...,...,...,...,...,...,...,...
643,lam_breadth=10000.0_lam_depth=10000.0_lam=0.1_i2,10000.0,10000.0,0.1,2,0.000004,1.788468e-06,1.052866,0.5481,10000.0,10000.0,0.1
644,lam_breadth=10000.0_lam_depth=10000.0_lam=0.2_i2,10000.0,10000.0,0.2,2,0.000004,1.220666e-06,0.822983,0.5295,10000.0,10000.0,0.2
645,lam_breadth=10000.0_lam_depth=10000.0_lam=0.3_i2,10000.0,10000.0,0.3,2,0.000003,1.340209e-06,0.562124,0.5043,10000.0,10000.0,0.3
646,lam_breadth=10000.0_lam_depth=10000.0_lam=0.5_i2,10000.0,10000.0,0.5,2,0.000002,7.148990e-07,0.276109,0.4769,10000.0,10000.0,0.5


In [6]:
lam_times = sorted(model_df["lam_time"].unique())
n_panels = len(lam_times)

zmin = model_df["accuracy"].min()
zmax = model_df["accuracy"].max()

fig = make_subplots(
    rows=1, cols=n_panels,
    horizontal_spacing=0.03,
    subplot_titles=[f"λ<sub>time</sub>={lt}" for lt in lam_times],
)

for j, lt in enumerate(lam_times):
    df_lt = model_df[model_df["lam_time"] == lt]
    pivot = df_lt.pivot_table(
        index="lam_depth",
        columns="lam_breadth",
        values="accuracy",
        aggfunc="mean",
    )

    fig.add_trace(
        go.Heatmap(
            z=pivot.values,
            x=[str(int(c)) for c in pivot.columns],
            y=[str(int(r)) for r in pivot.index],
            colorscale="Plasma",
            colorbar=dict(
                orientation="v", len=1.0, thickness=12, outlinewidth=0,
                title="accuracy",
            ) if j == n_panels - 1 else None,
            showscale=(j == n_panels - 1),
            text=pivot.values.round(2),
            texttemplate="%{text}",
            textfont_size=9,
            xgap=0, ygap=0,
            zmin=zmin,
            zmax=zmax,
            # zmin=0.87,
            # zmax=0.45,
        ),
        row=1, col=j + 1,
    )

    fig.update_xaxes(
        title_text="λ<sub>breadth</sub>", row=1, col=j + 1,
        type="category", categoryorder="array", categoryarray=BREADTH_TICK_VALS,
        tickmode="array", tickvals=BREADTH_TICK_VALS, ticktext=BREADTH_TICK_TEXT,
    )
    fig.update_yaxes(
        title_text="λ<sub>depth</sub>" if j == 0 else "",
        row=1, col=j + 1,
        type="category", categoryorder="array", categoryarray=DEPTH_TICK_VALS,
        tickmode="array", tickvals=DEPTH_TICK_VALS, ticktext=DEPTH_TICK_TEXT,
        showticklabels=(j == 0),
    )

fig.update_layout(
    height=WIDTH * 0.25,
    width=WIDTH,
    margin=dict(l=50, r=0, t=40, b=80),
)
fig.show()
nn_bowl.utils.save_figure(fig, figures_dir, "breadth_depth_time_accuracy_grids", scale=2)

In [7]:
# add all resources used
print("Loading pruning results and masks...")
pruning_results_dfs, pruning_masks = nn_bowl.utils.get_pruning_results_and_masks(cfg_base, model_df)

print("Loading time df...")
time_df = nn_bowl.utils.get_time_df(cfg_base, model_df)

# layers and channels used (vectorized)
print("Calculating layers and channels used...")
layers_used_list = []
channels_used_list = []
# for _, row in model_df.iterrows():
for _, row in tqdm.tqdm(model_df.iterrows(), total=len(model_df)):
    combo = {
        "costs.space.lam_breadth": row["lam_breadth"],
        "costs.space.lam_depth": row["lam_depth"],
        "costs.time.lam": row["lam_time"],
        "instance": row["instance"],
    }
    key = nn_bowl.utils.get_combo_key(combo)
    mask = pruning_masks.get(key)
    layer_sums = mask.sum(axis=1)
    active = layer_sums > 0
    layers = active.sum()
    channels = mask.sum() / layers if layers > 0 else 0
    layers_used_list.append(layers)
    channels_used_list.append(channels)
model_df["layers_used"] = layers_used_list
model_df["channels_used"] = channels_used_list

# time used (vectorized)
print("Calculating time used...")
time_used_df = time_df.pivot_table(
    index=[c for c in time_df.columns if c not in ("t", "t_weight")],
    columns="t",
    values="t_weight",
).reset_index()
time_used_df.columns = [f"t_{c}" if isinstance(c, (int, np.integer)) else c for c in time_used_df.columns]

t_cols = sorted(
    [c for c in time_used_df.columns if c.startswith("t_")],
    key=lambda c: int(c.split("_")[1]),
)
weights = np.arange(1, len(t_cols) + 1)
time_used_df["time_used"] = time_used_df[t_cols].fillna(0).values @ weights

time_used_agg = time_used_df.groupby(
    ["costs.space.lam_breadth", "costs.space.lam_depth", "costs.time.lam", "instance"]
)["time_used"].mean().reset_index()

model_df = model_df.merge(
    time_used_agg,
    on=["costs.space.lam_breadth", "costs.space.lam_depth", "costs.time.lam", "instance"],
    how="left",
)

Loading pruning results and masks...
Loading time df...


100%|██████████| 648/648 [00:14<00:00, 45.60it/s]


Calculating layers and channels used...


100%|██████████| 648/648 [00:00<00:00, 16730.12it/s]

Calculating time used...


In [8]:
model_df

,model_name,costs.space.lam_breadth,costs.space.lam_depth,costs.time.lam,instance,cost_depth,cost_breadth,cost_time,accuracy,lam_breadth,lam_depth,lam_time,layers_used,channels_used,time_used
0,lam_breadth=0.0_lam_depth=0.0_lam=0.0_i0,0.0,0.0,0.0,0,0.130335,1.372590e-01,1.221888,0.8759,0.0,0.0,0.0,6,47.166667,4.887551
1,lam_breadth=0.0_lam_depth=0.0_lam=0.1_i0,0.0,0.0,0.1,0,0.132233,1.377218e-01,0.993981,0.8829,0.0,0.0,0.1,6,47.666667,3.975924
2,lam_breadth=0.0_lam_depth=0.0_lam=0.2_i0,0.0,0.0,0.2,0,0.141518,1.425608e-01,0.677266,0.8739,0.0,0.0,0.2,6,47.500000,2.709065
3,lam_breadth=0.0_lam_depth=0.0_lam=0.3_i0,0.0,0.0,0.3,0,0.138716,1.410775e-01,0.621916,0.8713,0.0,0.0,0.3,6,47.666667,2.487665
4,lam_breadth=0.0_lam_depth=0.0_lam=0.5_i0,0.0,0.0,0.5,0,0.141579,1.363750e-01,0.467292,0.8642,0.0,0.0,0.5,6,47.833333,1.869168
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
643,lam_breadth=10000.0_lam_depth=10000.0_lam=0.1_i2,10000.0,10000.0,0.1,2,0.000004,1.788468e-06,1.052866,0.5481,10000.0,10000.0,0.1,2,5.500000,4.211464
644,lam_breadth=10000.0_lam_depth=10000.0_lam=0.2_i2,10000.0,10000.0,0.2,2,0.000004,1.220666e-06,0.822983,0.5295,10000.0,10000.0,0.2,2,6.000000,3.291932
645,lam_breadth=10000.0_lam_depth=10000.0_lam=0.3_i2,10000.0,10000.0,0.3,2,0.000003,1.340209e-06,0.562124,0.5043,10000.0,10000.0,0.3,1,10.000000,2.248498
646,lam_breadth=10000.0_lam_depth=10000.0_lam=0.5_i2,10000.0,10000.0,0.5,2,0.000002,7.148990e-07,0.276109,0.4769,10000.0,10000.0,0.5,2,4.500000,1.104435


In [9]:
import plotly.graph_objs as go
import plotly.offline as pyo

def pareto_indices(df, cost_x, cost_y, cost_z):
    data = df[[cost_x, cost_y, cost_z]].copy()
    data[cost_x] = -data[cost_x]
    data[cost_y] = -data[cost_y]
    data[cost_z] = -data[cost_z]
    data = data.values
    is_pareto = np.ones(data.shape[0], dtype=bool)
    for i, point in enumerate(data):
        if is_pareto[i]:
            mask = np.all(data <= point, axis=1) & np.any(data < point, axis=1)
            is_pareto[mask] = False
    return is_pareto

for acc in [0.7, 0.75, 0.8, 0.85]:
    pareto_df = model_df[model_df["accuracy"] >= acc].copy()
    pareto_mask = pareto_indices(pareto_df, "channels_used", "layers_used", "time_used")
    p = pareto_df[pareto_mask]
    print(f"\nAccuracy >= {acc} ({len(p)} Pareto models):")
    print(f"  channels: {p['channels_used'].min():.1f} – {p['channels_used'].max():.1f}")
    print(f"  layers:   {p['layers_used'].min():.1f} – {p['layers_used'].max():.1f}")
    print(f"  time:     {p['time_used'].min():.2f} – {p['time_used'].max():.2f}")

# for acc in [0.7, 0.75, 0.8, 0.85, 0.9]:
#     pareto_df = model_df[model_df["accuracy"] >= acc].copy()
#     pareto_mask = pareto_indices(pareto_df, "channels_used", "layers_used", "time_used")
#     pareto_df["is_pareto"] = pareto_mask

#     # count how many in pareto set
#     print(f"Models with accuracy above {acc}: {len(pareto_df)}, Pareto-optimal models: {pareto_mask.sum()}")


pareto_df = model_df[model_df["accuracy"] >= 0.7].copy()
pareto_mask = pareto_indices(pareto_df, "channels_used", "layers_used", "time_used")
pareto_df["is_pareto"] = pareto_mask

# count how many in pareto set
print(f"Total models: {len(pareto_df)}, Pareto-optimal models: {pareto_mask.sum()}")


fig = go.Figure()

from matplotlib.colors import to_rgba

for is_p, group in pareto_df.groupby("is_pareto"):
    # Normalize the "depth" axis to 0-1
    depth = group["time_used"]
    depth_norm = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)

    base_color = "red" if is_p else "gray"
    r, g, b, _ = to_rgba(base_color)

    # Map depth to brightness (0 = dark/far, 1 = full color/near)
    b_min, b_max = (0.0, 0.4) if is_p else (0.0, 0.8)
    brightnesses = b_min + depth_norm * (b_max - b_min)

    # Interpolate towards white: blend base color with white by brightness
    colors = [
        f"rgba({int((r + (1-r)*br)*255)},{int((g + (1-g)*br)*255)},{int((b + (1-b)*br)*255)},1)"
        for br in brightnesses
    ]
    
    
    fig.add_trace(go.Scatter3d(
        z=group["layers_used"],
        x=group["time_used"],
        y=group["channels_used"],
        mode="markers",
        marker=dict(
            size=8 if is_p else 5,
            color=colors,
            line=dict(width=0),
        ),
        name="Pareto" if is_p else "Non-Pareto",
    ))

fig.update_layout(
    height=WIDTH * 0.5, width=WIDTH * 0.5,
    margin=dict(l=0, r=0, t=0, b=0),
    scene=dict(
    zaxis=dict(
        title="layers used",
        tickvals=[1, 2, 3, 4, 5, 6],
        ticktext=["1", "2", "3", "4", "5", "6"],
    ),
    yaxis=dict(
        title="channels used",
        tickvals=[5, 10, 20, 30, 40],
        ticktext=["5", "10", "20", "30", "40"],
    ),
    xaxis=dict(
        title="time used",
        autorange="reversed",
        tickvals=[1, 2, 3, 4, 5],
        ticktext=["1", "2", "3", "4", "5"],
    ),
),
    # scene=dict(
    #     zaxis=dict(title="layers used"),
    #     yaxis=dict(title="channels used"),
    #     xaxis=dict(title="time used", autorange="reversed"),
    # ),
    scene_camera=dict(
        eye=dict(x=2.0, y=0.8, z=1.5)
    ),
)


pyo.iplot(fig)
fig.write_html(str(figures_dir / "breadth_depth_time_pareto_3d_scatter.html"))


Accuracy >= 0.7 (32 Pareto models):
  channels: 7.2 – 39.8
  layers:   2.0 – 6.0
  time:     1.10 – 4.89

Accuracy >= 0.75 (33 Pareto models):
  channels: 11.7 – 39.8
  layers:   2.0 – 6.0
  time:     1.10 – 4.89

Accuracy >= 0.8 (21 Pareto models):
  channels: 14.3 – 39.8
  layers:   3.0 – 6.0
  time:     1.10 – 4.84

Accuracy >= 0.85 (18 Pareto models):
  channels: 21.2 – 48.0
  layers:   3.0 – 6.0
  time:     1.10 – 4.89
Total models: 371, Pareto-optimal models: 32


In [10]:
scale = 2.5  # render at 2.5x, then shrink in LaTeX to 0.4 width

base_marker_p = 5
base_marker_np = 3
base_font = cfg_base.plotting.font.size
base_tick = 8
base_size = WIDTH * 0.4

fig = go.Figure()
for is_p, group in pareto_df.groupby("is_pareto"):
    depth = group["time_used"]
    depth_norm = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)
    base_color = "red" if is_p else "gray"
    r, g, b, _ = to_rgba(base_color)
    b_min, b_max = (0.0, 0.4) if is_p else (0.0, 0.8)
    brightnesses = b_min + depth_norm * (b_max - b_min)
    colors = [
        f"rgba({int((r + (1-r)*br)*255)},{int((g + (1-g)*br)*255)},{int((b + (1-b)*br)*255)},1)"
        for br in brightnesses
    ]
    fig.add_trace(go.Scatter3d(
        z=group["layers_used"],
        x=group["time_used"],
        y=group["channels_used"],
        mode="markers",
        marker=dict(
            size=(base_marker_p if is_p else base_marker_np) * scale,
            color=colors,
            line=dict(width=0),
        ),
        name="Pareto" if is_p else "Non-Pareto",
    ))

fig.update_layout(
    height=base_size * scale, width=base_size * scale,
    margin=dict(l=0, r=0, t=0, b=0),
    scene=dict(
        zaxis=dict(
            title=dict(text="layers used", font=dict(size=base_font * scale)),
            tickvals=[1, 2, 3, 4, 5, 6],
            ticktext=["1", "2", "3", "4", "5", "6"],
            tickfont=dict(size=base_tick * scale),
        ),
        yaxis=dict(
            title=dict(text="channels used", font=dict(size=base_font * scale)),
            tickvals=[10, 20, 30, 40],
            ticktext=["10", "20", "30", "40"],
            tickfont=dict(size=base_tick * scale),
        ),
        xaxis=dict(
            title=dict(text="time used", font=dict(size=base_font * scale)),
            autorange="reversed",
            tickvals=[1, 2, 3, 4, 5],
            ticktext=["1", "2", "3", "4", "5"],
            tickfont=dict(size=base_tick * scale),
        ),
    ),
    scene_camera=dict(eye=dict(x=2.0, y=0.8, z=1.5)),
    legend=dict(font=dict(size=base_tick * scale)),
)
fig.update_layout(
    scene=dict(
        annotations=[
            dict(
                showarrow=False,
                text="<b>Pareto</b>",
                font=dict(size=base_font * scale, color="red"),
                x=2, y=5, z=6,  # position near a Pareto point
                # arrowcolor="red",
                # arrowsize=2,
                # arrowwidth=1,
            ),
        ],
    ),
)

fig.show()
fig.write_html(str(figures_dir / "breadth_depth_time_pareto_3d_scatter.html"))

In [11]:
# 2D pairwise projections
pairs = [
    ("channels_used", "layers_used", "channels used", "layers used"),
    ("time_used", "layers_used", "time used", "layers used"),
    ("time_used", "channels_used", "time used", "channels used"),
]

fig2d = make_subplots(rows=1, cols=3, horizontal_spacing=0.12)

for j, (xcol, ycol, xlabel, ylabel) in enumerate(pairs):
    # non-pareto
    fig2d.add_trace(go.Scatter(
        x=pareto_df.loc[~pareto_mask, xcol],
        y=pareto_df.loc[~pareto_mask, ycol],
        mode='markers',
        marker=dict(color='lightgrey', size=5, opacity=0.5),
        showlegend=False,
    ), row=1, col=j+1)

    # pareto
    fig2d.add_trace(go.Scatter(
        x=pareto_df.loc[pareto_mask, xcol],
        y=pareto_df.loc[pareto_mask, ycol],
        mode='markers',
        marker=dict(color='red', size=8, opacity=0.9,
                    line=dict(width=0.5, color='white')),
        showlegend=(j == 0),
        name='Pareto',
    ), row=1, col=j+1)

    fig2d.update_xaxes(title_text=xlabel, row=1, col=j+1)
    fig2d.update_yaxes(title_text=ylabel if j == 0 else ylabel, row=1, col=j+1)

fig2d.update_layout(
    height=WIDTH * 0.3,
    width=WIDTH * 0.7,
    margin=dict(l=50, r=10, t=20, b=80),
)
fig2d.show()
nn_bowl.utils.save_figure(fig2d, figures_dir, "breadth_depth_time_pareto_2d_projections", scale=2)

In [12]:
# probs for each row in the model_df
probs = nn_bowl.utils.load_all_probs(cfg_base, model_df)

loading all probs based on existing rows in model df..


100%|██████████| 648/648 [00:26<00:00, 24.02it/s]


In [13]:
# Compute JS divergence distance matrix
n_models = probs.shape[0]
distances = np.zeros((n_models, n_models))

for i in tqdm.tqdm(range(n_models)):
    for j in range(i+1, n_models):
        dist = nn_bowl.utils.js_divergence_fast(probs[i], probs[j])
        distances[i, j] = dist
        distances[j, i] = dist

print("Distance matrix computed!")
print(distances)

  0%|          | 0/648 [00:00<?, ?it/s]

100%|██████████| 648/648 [02:51<00:00,  3.79it/s]

Distance matrix computed!
[[0.         0.23924574 0.25241661 ... 0.53122699 0.53871244 0.57558912]
 [0.23924574 0.         0.24722904 ... 0.52858239 0.53634942 0.57331419]
 [0.25241661 0.24722904 0.         ... 0.54093373 0.54801917 0.584288  ]
 ...
 [0.53122699 0.52858239 0.54093373 ... 0.         0.24535161 0.25197822]
 [0.53871244 0.53634942 0.54801917 ... 0.24535161 0.         0.22454689]
 [0.57558912 0.57331419 0.584288   ... 0.25197822 0.22454689 0.        ]]


In [14]:
probs.shape

(648, 10000, 10)

In [15]:
all_entropies = np.sum(-probs * np.log(probs + 1e-10), axis=2)  # (n_models, n_images)
model_entropies = all_entropies.mean(axis=1)  # (n_models,) average entropy per model
model_df["entropy"] = model_entropies
model_df

,model_name,costs.space.lam_breadth,costs.space.lam_depth,costs.time.lam,instance,cost_depth,cost_breadth,cost_time,accuracy,lam_breadth,lam_depth,lam_time,layers_used,channels_used,time_used,entropy
0,lam_breadth=0.0_lam_depth=0.0_lam=0.0_i0,0.0,0.0,0.0,0,0.130335,1.372590e-01,1.221888,0.8759,0.0,0.0,0.0,6,47.166667,4.887551,0.247198
1,lam_breadth=0.0_lam_depth=0.0_lam=0.1_i0,0.0,0.0,0.1,0,0.132233,1.377218e-01,0.993981,0.8829,0.0,0.0,0.1,6,47.666667,3.975924,0.252479
2,lam_breadth=0.0_lam_depth=0.0_lam=0.2_i0,0.0,0.0,0.2,0,0.141518,1.425608e-01,0.677266,0.8739,0.0,0.0,0.2,6,47.500000,2.709065,0.211845
3,lam_breadth=0.0_lam_depth=0.0_lam=0.3_i0,0.0,0.0,0.3,0,0.138716,1.410775e-01,0.621916,0.8713,0.0,0.0,0.3,6,47.666667,2.487665,0.250092
4,lam_breadth=0.0_lam_depth=0.0_lam=0.5_i0,0.0,0.0,0.5,0,0.141579,1.363750e-01,0.467292,0.8642,0.0,0.0,0.5,6,47.833333,1.869168,0.262026
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
643,lam_breadth=10000.0_lam_depth=10000.0_lam=0.1_i2,10000.0,10000.0,0.1,2,0.000004,1.788468e-06,1.052866,0.5481,10000.0,10000.0,0.1,2,5.500000,4.211464,1.394003
644,lam_breadth=10000.0_lam_depth=10000.0_lam=0.2_i2,10000.0,10000.0,0.2,2,0.000004,1.220666e-06,0.822983,0.5295,10000.0,10000.0,0.2,2,6.000000,3.291932,1.482944
645,lam_breadth=10000.0_lam_depth=10000.0_lam=0.3_i2,10000.0,10000.0,0.3,2,0.000003,1.340209e-06,0.562124,0.5043,10000.0,10000.0,0.3,1,10.000000,2.248498,1.587262
646,lam_breadth=10000.0_lam_depth=10000.0_lam=0.5_i2,10000.0,10000.0,0.5,2,0.000002,7.148990e-07,0.276109,0.4769,10000.0,10000.0,0.5,2,4.500000,1.104435,1.496650


In [16]:
model_names = [f"d{d}_b{b}_t{t}_i{i}" for d, b, t, i in model_df[["lam_depth", "lam_breadth", "lam_time", "instance"]].values]

accuracies = model_df["accuracy"].values
entropies = model_df["entropy"].values

# Plot 1: Heatmap
fig_heatmap = go.Figure(data=go.Heatmap(
    z=distances,
    x=model_names,
    y=model_names,
    colorscale='Viridis',
    colorbar=dict(title='JS Divergence'),
    text=np.round(distances, 3) if n_models < 15 else None,
    texttemplate='%{text:.3f}' if n_models < 15 else None,
))
fig_heatmap.update_layout(
    title='Pairwise JS Divergence Between Models',
    width=WIDTH * 0.5,
    height=WIDTH * 0.5,
)
fig_heatmap.show()

# Compute MDS embedding
mds = MDS(n_components=2, dissimilarity='precomputed', random_state=42)
embedding = mds.fit_transform(distances)

def make_log_safe(vals):
    vals_safe = vals.copy()
    min_nonzero = vals[vals > 0].min() if np.any(vals > 0) else 1e-10
    vals_safe[vals_safe == 0] = min_nonzero / 10
    return vals_safe



In [17]:
from sklearn.linear_model import LinearRegression

# fit a line: accuracy as a function of MDS coordinates
reg = LinearRegression().fit(embedding, accuracies)
acc_direction = reg.coef_ / np.linalg.norm(reg.coef_)

# orthogonal direction
orth_direction = np.array([-acc_direction[1], acc_direction[0]])

# rotate embedding
rotation_matrix = np.stack([orth_direction, acc_direction])
embedding_rotated = embedding @ rotation_matrix.T

In [18]:

channels_used = model_df["channels_used"].values
layers_used = model_df["layers_used"].values
time_used = model_df["time_used"].values
accuracies = model_df["accuracy"].values
# entropies = model_df["entropy"].values

cmaps = {
    "channels_used" : [[0, "#f7f7f7"], [1, cfg_base.plotting.breadth]],
    "layers_used" : [[0, "#f7f7f7"], [1, cfg_base.plotting.depth]],
    "time_used" : [[0, "#f7f7f7"], [1, cfg_base.plotting.time]],
}
# import plotly.colors as pc

# def make_cmap(hex_color):
#     r, g, b = int(hex_color[1:3], 16), int(hex_color[3:5], 16), int(hex_color[5:7], 16)
#     light = f"rgb({min(r+180, 255)}, {min(g+180, 255)}, {min(b+180, 255)})"
#     dark = f"rgb({r}, {g}, {b})"
#     return [[0, light], [1, dark]]

# cmaps = {
#     "channels_used": make_cmap(cfg_base.plotting.breadth),
#     "layers_used": make_cmap(cfg_base.plotting.depth),
#     "time_used": make_cmap(cfg_base.plotting.time),
# }

# Plot 2: MDS scatter plots
params = [
    # ('λ_depth', lam_depth_vals, 'Reds', True),
    # ('λ_breadth', lam_breadth_vals, 'Greens', True),
    # ('λ_time', lam_time_vals, 'Blues', True),
    ('channels used', 'channels<br>used', channels_used, cmaps["channels_used"]),
    ('layers used', 'layers<br>used', layers_used, cmaps["layers_used"]),
    ('time used', 'time<br>used', time_used, cmaps["time_used"]),
    # ('entropy', 'entropy', entropies, 'gray'),
    ('accuracy', 'accuracy', accuracies, 'Plasma'),
]

fig_mds = make_subplots(rows=1, cols=4, subplot_titles=[], horizontal_spacing=0.15, vertical_spacing=0.2)

for i, (name, cbar_name, vals, cmap) in enumerate(params):
    row = i // 4 + 1
    col = i % 4 + 1

    axis_suffix = '' if i == 0 else str(i + 1)
    xaxis = fig_mds.layout[f'xaxis{axis_suffix}']
    yaxis = fig_mds.layout[f'yaxis{axis_suffix}']

    fig_mds.add_trace(
        go.Scatter(
            # x=embedding[:, 0],
            # y=embedding[:, 1],
            x=embedding_rotated[:, 0],
            y=embedding_rotated[:, 1],
            mode='markers',
            marker=dict(
                size=4,
                color=vals,
                colorscale=cmap,
                # layers used: pad the top of the color range so the "6" tick label clears the colorbar title
                cmin=vals.min(), cmax=vals.max() + (0.35 if name == "layers used" else 0),
                colorbar=dict(
                    title=dict(text=cbar_name, font=dict(size=15)),
                    thickness=10,
                    # len=(yaxis.domain[1] - yaxis.domain[0]),
                    len=0.8,
                    x=xaxis.domain[1] + 0.01,
                    y=(yaxis.domain[0] + yaxis.domain[1]) / 2,
                ),
                # line=dict(width=0.5, color='black'),
                opacity=1.0,
            ),
            text=[f"{name}: {v:.4f}<br>Model: {m}" for v, m in zip(vals, model_names)],
            hovertemplate='%{text}<extra></extra>',
            showlegend=False,
        ),
        row=row, col=col,
    )
    fig_mds.update_xaxes(title_text='MDS 1', row=row, col=col, title_standoff=2)
    fig_mds.update_yaxes(title_text='MDS 2', row=row, col=col, title_standoff=2)

fig_mds.update_layout(
    width=WIDTH,
    height=WIDTH * 0.25,
    margin=dict(l=50, r=10, t=10, b=40),
)
fig_mds.show()

nn_bowl.utils.save_figure(fig_mds, figures_dir, "breadth_depth_time_mds_scatter", scale=2)

In [19]:
# error consistency
def error_consistency(probs1, probs2, labels):
    preds1 = np.argmax(probs1, axis=1)
    preds2 = np.argmax(probs2, axis=1)

    correct1 = (preds1 == labels)
    correct2 = (preds2 == labels)


    acc1 = np.mean(correct1)
    acc2 = np.mean(correct2)

    consistency_exp = acc1 * acc2 + (1 - acc1) * (1 - acc2)
    consistency_obs = np.mean(correct1 == correct2)

    error_consistency = (consistency_obs - consistency_exp) / (1 - consistency_exp + 1e-10)
    return error_consistency


error_consistencies = np.zeros((n_models, n_models))
labels = []
nn_bowl.utils.get_dataloaders(cfg_base)["test"]
for data in nn_bowl.utils.get_dataloaders(cfg_base)["test"]:
    labels.append(data["label"].numpy())
labels = np.concatenate(labels)


for i in tqdm.tqdm(range(n_models)):
    for j in range(i+1, n_models):
        ec = error_consistency(probs[i], probs[j], labels)
        error_consistencies[i, j] = ec
        error_consistencies[j, i] = ec


100%|██████████| 648/648 [02:18<00:00,  4.68it/s]


In [20]:
combo_cols = ["lam_depth", "lam_breadth", "lam_time"]
combo_groups = model_df.groupby(combo_cols)

combo_keys = list(combo_groups.groups.keys())
combo_indices = {k: g.index.values for k, g in combo_groups}
n_configs = len(combo_keys)

combo_ec = np.zeros((n_configs, n_configs))
for i in range(n_configs):
    for j in range(i + 1, n_configs):
        ecs = []
        for ii in combo_indices[combo_keys[i]]:
            for jj in combo_indices[combo_keys[j]]:
                ecs.append(error_consistencies[ii, jj])
        combo_ec[i, j] = np.mean(ecs)
        combo_ec[j, i] = combo_ec[i, j]

combo_df = pd.DataFrame(combo_keys, columns=combo_cols)
combo_df

,lam_depth,lam_breadth,lam_time
0,0.0,0.0,0.0
1,0.0,0.0,0.1
2,0.0,0.0,0.2
3,0.0,0.0,0.3
4,0.0,0.0,0.5
...,...,...,...
211,10000.0,10000.0,0.1
212,10000.0,10000.0,0.2
213,10000.0,10000.0,0.3
214,10000.0,10000.0,0.5


In [21]:

print(f"configs: {len(combo_df)}, models: {len(model_df)}")

configs: 216, models: 648


In [22]:
tick_lookup = {
    "lam_breadth": (BREADTH_TICK_VALS, BREADTH_TICK_TEXT),
    "lam_depth": (DEPTH_TICK_VALS, DEPTH_TICK_TEXT),
    "lam_time": (TIME_TICK_VALS, TIME_TICK_TEXT),
}


ec_min = combo_ec.min()
ec_max = combo_ec.max()
mid = (ec_min + ec_max)  * 0.55

ec_cmaps = {
    "lam_breadth": [[0, "#ffffff"], [mid, "#ffffff"], [1, cfg_base.plotting.breadth]],
    "lam_depth": [[0, "#ffffff"], [mid, "#ffffff"], [1, cfg_base.plotting.depth]],
    "lam_time": [[0, "#ffffff"], [mid, "#ffffff"], [1, cfg_base.plotting.time]],
}

axis_labels = {
    "lam_breadth": "λ<sub>breadth</sub>",
    "lam_depth": "λ<sub>depth</sub>",
    "lam_time": "λ<sub>time</sub>",
}
sort_configs = [
    ("sorted by λ<sub>breadth</sub>", "lam_breadth", ["lam_breadth", "lam_depth", "lam_time"]),
    ("sorted by λ<sub>depth</sub>", "lam_depth", ["lam_depth", "lam_breadth", "lam_time"]),
    ("sorted by λ<sub>time</sub>", "lam_time", ["lam_time", "lam_depth", "lam_breadth"]),
]

fig_heatmaps = make_subplots(
    rows=1, cols=3,
    subplot_titles=[title for title, _, _ in sort_configs],
    horizontal_spacing=0.15,
)

for col, (title, primary_key, sort_by) in enumerate(sort_configs, start=1):
    # df_sorted = combo_df.sort_values(sort_by).reset_index(drop=False)
    # Sort by primary key only, then shuffle within each group
    df_sorted = combo_df.copy()
    df_sorted['_rand'] = np.random.default_rng(42).permutation(len(df_sorted))
    df_sorted = df_sorted.sort_values([primary_key, '_rand']).reset_index(drop=False)
    df_sorted = df_sorted.drop(columns='_rand')

    sort_idx = df_sorted["index"].values
    ec_sorted = combo_ec[np.ix_(sort_idx, sort_idx)]
    n = len(sort_idx)

    sorted_vals = df_sorted[primary_key].values
    tick_vals_list, tick_text_list = tick_lookup[primary_key]

    group_ranges = []
    for val, text in zip(tick_vals_list, tick_text_list):
        indices = np.where(sorted_vals == val)[0]
        if len(indices) > 0:
            group_ranges.append((indices[0], indices[-1], text))

    axis_suffix = '' if col == 1 else str(col)
    xaxis = fig_heatmaps.layout[f'xaxis{axis_suffix}']
    yaxis = fig_heatmaps.layout[f'yaxis{axis_suffix}']
    x_dom = xaxis.domain
    y_dom = yaxis.domain

    fig_heatmaps.add_trace(
        go.Heatmap(
            z=ec_sorted,
            x=np.arange(n),
            y=np.arange(n),
            colorscale=ec_cmaps[primary_key],
            zmin=ec_min,
            # zmin=0.0,
            zmax=ec_max,
            # zmax=0.6,
            colorbar=dict(
                title=dict(text='error<br>consistency',
                        #    font=dict(size=cfg_base.plotting.font.size)),
                           font=dict(size=15)),
                len=1.0,
                thickness=12,
                x=x_dom[1] + 0.0,
                y=(y_dom[0] + y_dom[1]) * 0.6,
                yanchor='middle',
            ),
            showscale=True,
        ),
        row=1, col=col,
    )

    for start, end, text in group_ranges:
        # Center of group in data coords, mapped to paper coords
        group_center = (start + end) / 2
        x_center = x_dom[0] + (group_center / (n - 1)) * (x_dom[1] - x_dom[0])

        # Bracket edges aligned to cell edges
        x_left = x_dom[0] + ((start - 0.5) / n) * (x_dom[1] - x_dom[0])
        x_right = x_dom[0] + ((end + 0.5) / n) * (x_dom[1] - x_dom[0])
        x_bracket_mid = (x_left + x_right) / 2

        y_top = y_dom[0] - 0.02
        y_bot = y_dom[0] - 0.05

        fig_heatmaps.add_shape(
            type="path",
            path=f"M {x_left},{y_top} L {x_left},{y_bot} L {x_bracket_mid},{y_bot} "
                 f"M {x_bracket_mid},{y_bot} L {x_right},{y_bot} L {x_right},{y_top}",
            line=dict(color="black", width=1),
            xref="paper", yref="paper",
        )

        fig_heatmaps.add_annotation(
            x=x_bracket_mid, y=y_bot - 0.18,
            xref='paper', yref='paper',
            text=text,
            font=dict(size=cfg_base.plotting.font.size),
            showarrow=False,
            yanchor='bottom',
            xanchor='center',
        )

    fig_heatmaps.update_layout(**{
        f'xaxis{axis_suffix}': dict(
            showticklabels=False,
            ticks='',
            showgrid=False,
            zeroline=False,
            title_standoff=50,
            title_text=axis_labels[primary_key],
        ),
        f'yaxis{axis_suffix}': dict(
            tickvals=[r[0] + (r[1] - r[0]) / 2 for r in group_ranges],
            ticktext=[r[2] for r in group_ranges],
        ),
    })

fig_heatmaps.update_layout(
    width=WIDTH,
    height=WIDTH * 0.3,
    margin=dict(l=50, r=10, t=40, b=80),
)

fig_heatmaps.show()

nn_bowl.utils.save_figure(fig_heatmaps, figures_dir, "breadth_depth_time_error_consistency_heatmaps", scale=2)

In [23]:
# RSA

